# RLVR Base 5k — GRPO on gemma-4-E2B (no SFT) — 2-step gate + A1/B 200-step

Pool: results/rlvr-pool/train-5k.jsonl (5k stratified 30/40/30) — base gemma, P1 endorsed-only, Dr.GRPO, G8 256, HF checkpoints every 300s. Push via `kaggle kernels push -p notebooks/push_rlvr_base_5k`

In [ ]:
import os, subprocess, pathlib
repo = "https://github.com/Vedang-P/chess-slm-benchmark.git"
if not pathlib.Path("chess-slm-benchmark").exists():
    subprocess.run(["git", "clone", repo], check=True)
%cd chess-slm-benchmark
subprocess.run(["git", "pull"], check=True)
print("repo ready:", pathlib.Path.cwd())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.54.*", "trl==0.17.*", "peft==0.15.*", "bitsandbytes==0.46.*", "huggingface_hub", "python-chess", "datasets"], check=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "stockfish"], check=True)
import shutil
print("stockfish:", shutil.which("stockfish") or "/usr/games/stockfish")

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi
import os
# The 5k pool is built locally and pushed to HF dataset under rlvr-pool/train-5k.jsonl
# For now, if not on HF, expect it at data/positions or results/rlvr-pool
pool_src = Path("results/rlvr-pool/train-5k.jsonl")
if not pool_src.exists():
    # fallback: try HF dataset
    from huggingface_hub import hf_hub_download
    try:
        pool_src = Path(hf_hub_download(repo_id="vedangfake/chess-slm-benchmark", repo_type="dataset", filename="rlvr-pool/train-5k.jsonl"))
        print("pool from HF:", pool_src)
    except Exception as e:
        print("pool not found:", e)
        raise
pool_dst = Path("/kaggle/working/pool.jsonl")
pool_dst.write_bytes(pool_src.read_bytes())
print("pool ready:", pool_dst, len(pool_dst.read_text().splitlines()), "rows")
# preview
import json
print(json.loads(pool_dst.read_text().splitlines()[0]))

In [ ]:
import os
# HF token from Kaggle secret or env (not hardcoded for GitHub push protection)
# On Kaggle, set HF_WRITE_TOKEN as secret or env var; fallback to .env locally
if 'HF_WRITE_TOKEN' not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_WRITE_TOKEN'] = UserSecretsClient().get_secret('HF_WRITE_TOKEN')
    except Exception:
        pass
print('HF token set', len(os.environ.get('HF_WRITE_TOKEN','')))


In [ ]:
import os, signal, subprocess, sys
# 2-step gate on base gemma-4-E2B (no SFT merge) — must show MoveA: and outcome>0
cmd = [sys.executable, "scripts/train_mate_grpo.py",
       "--base", "google/gemma-2-2b-it",
       "--train", "/kaggle/working/pool.jsonl",
       "--out", "/kaggle/working/rlvr-gate-adapter",
       "--oracle", "stockfish", "--stockfish", "/usr/games/stockfish",
       "--depth", "12",
       "--max-steps", "2", "--group", "8",
       "--max-completion-length", "256",
       "--temperature", "0.7", "--top-p", "0.9",
       "--save-steps", "1",
       "--hf-repo", "vedangfake/chess-slm-benchmark", "--hf-tag", "rlvr-gate-base5k",
       "--hf-upload-every", "60",
       "--progress-every", "60",
       "--step-timeout-min", "45"]
print("gate running:", " ".join(cmd))
proc = subprocess.Popen(cmd)
try:
    rc = proc.wait(timeout=600)
except subprocess.TimeoutExpired:
    proc.send_signal(signal.SIGINT)
    rc = proc.wait(timeout=90)
if rc != 0:
    raise SystemExit(f"gate failed: {rc}")
print("gate done — check logs for MoveA: and outcome_reward mean>0")

In [ ]:
import subprocess, sys, signal
# Arm A1: 1.0 outcome +0.3 process +0.0 style, P1, Dr.GRPO, G8 256
cmd = [sys.executable, "scripts/train_mate_grpo.py",
       "--base", "google/gemma-2-2b-it",
       "--train", "/kaggle/working/pool.jsonl",
       "--out", "/kaggle/working/rlvr-a1-adapter",
       "--oracle", "stockfish", "--stockfish", "/usr/games/stockfish",
       "--depth", "12",
       "--max-steps", "200", "--group", "8",
       "--max-completion-length", "256",
       "--temperature", "0.7", "--top-p", "0.9",
       "--save-steps", "25",
       "--hf-repo", "vedangfake/chess-slm-benchmark", "--hf-tag", "rlvr-a1-base5k",
       "--hf-upload-every", "300",
       "--progress-every", "60",
       "--step-timeout-min", "45",
       "--wandb-project", "chess-slm-rlvr"]
print("A1 running:", " ".join(cmd))
proc = subprocess.Popen(cmd)
try:
    rc = proc.wait(timeout=720*60)
except subprocess.TimeoutExpired:
    proc.send_signal(signal.SIGINT)
    rc = proc.wait(timeout=90)
if rc != 0:
    raise SystemExit(f"A1 failed: {rc}")
print("A1 done")